In [1]:
# install required packages
%pip install peewee pandas


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Database intro 

In [2]:
# importing the library
from peewee import *

In [3]:
# defining our database and giving it a name
db = SqliteDatabase('maryland_athletics.db')

# define the base model 
# this connects every model we define to the same database.
class BaseModel(Model):
    class Meta:
        database = db

## Let's create our first table!

In [4]:
class Team(BaseModel):
    name = CharField(unique=True)
    sport = CharField()
    season = CharField()
    class Meta:
        database = db

In [5]:
# connecting to our maryland_athletics.db database
db.connect()
# telling it to create the team table, which we've defined.
db.create_tables([Team])

In [6]:
# Let's take a look at this table. Is there anything in it?
for team in Team.select():
    print(team.name)

Maryland Women's Basketball
Maryland Men's Soccer
Maryland Football


In [7]:
# insert data into our table 
wbb_25, created = Team.get_or_create(
    name="Maryland Women's Basketball",
    defaults={
        "sport": "Women's Basketball",
        "season": "2025"
    }
)

msoc_25, created = Team.get_or_create(
    name="Maryland Men's Soccer",
    defaults={
        "sport": "Men's Soccer",
        "season": "2025"
    }
)

fb_25, created = Team.get_or_create(
    name="Maryland Football",
    defaults={
        "sport": "Football",
        "season": "2025"
    }
)

In [8]:
for team in Team.select():
    print(team.name)

Maryland Women's Basketball
Maryland Men's Soccer
Maryland Football


In [9]:
class Athlete(BaseModel):
    name = CharField()
    team = ForeignKeyField(Team, backref='athletes')
    athlete_year = CharField()
    athlete_position = CharField()
    class Meta:
        database = db

db.create_tables([Athlete])

In [10]:
malik_washington, created = Athlete.get_or_create(
    name="Malik Washington",
    defaults={
        "team": fb_25,
        "athlete_year": "Junior",
        "athlete_position": "Wide Receiver"
    }
)

In [11]:
for team in Team.select():
    print(team.name)

Maryland Women's Basketball
Maryland Men's Soccer
Maryland Football


In [12]:
class Game(BaseModel):
    date = DateField()
    opponent = CharField()
    home_away = CharField()     
    team = ForeignKeyField(Team, backref='games')
    class Meta:
        database = db
    

# stats are sport-agnostic — null fields just won't apply to every sport
class Stat(BaseModel):
    game = ForeignKeyField(Game, backref='stats')
    athlete = ForeignKeyField(Athlete, backref='stats')
    minutes_played = IntegerField(null=True)
    points = IntegerField(null=True)
    rebounds = IntegerField(null=True)
    assists = IntegerField(null=True)
    steals = IntegerField(null=True)
    turnovers = IntegerField(null=True)
    blocks = IntegerField(null=True)
    goals = IntegerField(null=True)
    shots = IntegerField(null=True)
    saves = IntegerField(null=True)
    class Meta:
        database = db

db.create_tables([Game, Stat])

In [13]:
# close the database connection when we're done
db.close()

True

## Now let's go over adding data using a spreadsheet and turning that into a database!


In [14]:
# read the spreadsheet
import pandas as pd
mcap_24 = pd.read_csv('data/data-9Tdi0.csv')

In [15]:
mcap_24

,District,School,Assessment,"Proficient, 2022","Proficient, 2023","Proficient, 2024",Percentage point change 2022-'23,Percentage point change 2023-'24
0,Montgomery,A. Mario Loiederman Middle,Mathematics 6,6,9.9,9.5,3.9,-0.4
1,Montgomery,A. Mario Loiederman Middle,Algebra 1,5,5,14,NaN,NaN
2,Montgomery,A. Mario Loiederman Middle,Geometry,5.7,9.3,14.3,3.6,5.0
3,Montgomery,A. Mario Loiederman Middle,English Language Arts 8,35.2,28.8,31.4,-6.4,2.6
4,Montgomery,A. Mario Loiederman Middle,English Language Arts 6,32.6,35.8,37.7,3.2,1.9
...,...,...,...,...,...,...,...,...
9086,Harford,Youths Benefit Elementary,Mathematics 4,47.8,52.4,56,4.6,3.6
9087,Harford,Youths Benefit Elementary,Mathematics 3,66.1,68,56,1.9,-12.0
9088,Harford,Youths Benefit Elementary,English Language Arts 5,58.8,59.2,67.6,0.4,8.4
9089,Harford,Youths Benefit Elementary,English Language Arts 3,66.9,70.7,67.9,3.8,-2.8


In [16]:
# get rid of the rows where district is called "seed" — these are just the average proficiency values for the state, not actual districts
mcap_24 = mcap_24[~mcap_24["District"].str.lower().eq("seed")]

## Clean up the data

In [17]:
# make the data long
long_df = mcap_24.melt(
    id_vars=["District", "School", "Assessment"],
    value_vars=[
        "Proficient, 2022",
        "Proficient, 2023",
        "Proficient, 2024"
    ],
    var_name="year",
    value_name="proficient_percent"
)

In [18]:
long_df

,District,School,Assessment,year,proficient_percent
0,Montgomery,A. Mario Loiederman Middle,Mathematics 6,"Proficient, 2022",6
1,Montgomery,A. Mario Loiederman Middle,Algebra 1,"Proficient, 2022",5
2,Montgomery,A. Mario Loiederman Middle,Geometry,"Proficient, 2022",5.7
3,Montgomery,A. Mario Loiederman Middle,English Language Arts 8,"Proficient, 2022",35.2
4,Montgomery,A. Mario Loiederman Middle,English Language Arts 6,"Proficient, 2022",32.6
...,...,...,...,...,...
27238,Harford,Youths Benefit Elementary,Mathematics 4,"Proficient, 2024",56
27239,Harford,Youths Benefit Elementary,Mathematics 3,"Proficient, 2024",56
27240,Harford,Youths Benefit Elementary,English Language Arts 5,"Proficient, 2024",67.6
27241,Harford,Youths Benefit Elementary,English Language Arts 3,"Proficient, 2024",67.9


In [19]:
# create a new year column based on the old year column
long_df["year"] = long_df["year"].astype(str).str.extract(r'(\d{4})').astype(int)

In [20]:
long_df["proficient_percent"] = (
    long_df["proficient_percent"]
    .replace("*", None)
    .astype(float)
)

In [21]:
long_df

,District,School,Assessment,year,proficient_percent
0,Montgomery,A. Mario Loiederman Middle,Mathematics 6,2022,6.0
1,Montgomery,A. Mario Loiederman Middle,Algebra 1,2022,5.0
2,Montgomery,A. Mario Loiederman Middle,Geometry,2022,5.7
3,Montgomery,A. Mario Loiederman Middle,English Language Arts 8,2022,35.2
4,Montgomery,A. Mario Loiederman Middle,English Language Arts 6,2022,32.6
...,...,...,...,...,...
27238,Harford,Youths Benefit Elementary,Mathematics 4,2024,56.0
27239,Harford,Youths Benefit Elementary,Mathematics 3,2024,56.0
27240,Harford,Youths Benefit Elementary,English Language Arts 5,2024,67.6
27241,Harford,Youths Benefit Elementary,English Language Arts 3,2024,67.9


In [22]:
# drop rows with missing proficiency values
long_df = long_df.dropna(subset=["proficient_percent"])

In [23]:
# create a new year column based on the old year column
long_df["year"] = long_df["year"].astype(str).str.extract(r'(\d{4})').astype(int)

In [24]:
# create a new database for the mcap data
db = SqliteDatabase("mcap.db")

# define the schemas for all of our tables!
class BaseModel(Model):
    class Meta:
        database = db
# district table
class District(BaseModel):
    name = CharField(unique=True)
# school table
class School(BaseModel):
    name = CharField()
    district = ForeignKeyField(District, backref="schools")

    class Meta:
        indexes = (
            (("name", "district"), True),
        )
# assessment table
class Assessment(BaseModel):
    name = CharField(unique=True)
# result table
class Result(BaseModel):
    school = ForeignKeyField(School, backref="results")
    assessment = ForeignKeyField(Assessment, backref="results")
    year = IntegerField()
    proficient_percent = FloatField()

    class Meta:
        indexes = (
            (("school", "assessment", "year"), True),
        )

In [25]:
# create the tables
db.connect()
db.create_tables([District, School, Assessment, Result])

In [26]:
# insert districts
for district_name in long_df["District"].unique():
    District.get_or_create(name=district_name)
# insert schools
for _, row in long_df[["School", "District"]].drop_duplicates().iterrows():
    district = District.get(District.name == row["District"])

    School.get_or_create(
        name=row["School"],
        district=district
    )
# insert assessments
for assessment_name in long_df["Assessment"].unique():
    Assessment.get_or_create(name=assessment_name)
# insert results
for _, row in long_df.iterrows():
    school = School.get(School.name == row["School"])
    assessment = Assessment.get(Assessment.name == row["Assessment"])

    Result.get_or_create(
        school=school,
        assessment=assessment,
        year=row["year"],
        defaults={"proficient_percent": row["proficient_percent"]}
    )

# Your turn!!!
For your homework, create a relational database for any data of your choice. There must be at least three tables and at least 10 rows in each table. You may either manually add the data or turn a spreadsheet into a relational database. Use the peewee package we went over in class. LLMS/AI are your friend! Use GitHub Copilot if you get stuck.

In [27]:
from peewee import *

# defining our database and giving it a name
dbk = SqliteDatabase('dbk.db')

# define the base model 
# this connects every model we define to the same database.
class BaseModel(Model):
    class Meta:
        database = dbk

In [28]:
import pandas as pd
dbk_stories = pd.read_csv('data/dbkstories - Sheet1.csv')

In [29]:
dbk_stories

,Article,Date,Writer,Editor,Tag 1,Tag 2,Tag 3,Tage 4,Tag 5,Category
0,UMD Dining Services tracks student food waste ...,3/10/2026,Johana Gonzalez-Cruz,Pera Onal,251 north,composting,umd dining services,food waste,recycling,Campus
1,Old College Park apartment complex reopens as ...,3/10/2026,Amelia Twyman,Natalie Weger,apartment,apartment building,college park housing,the arbor,leasing,Local
2,UMD students voice frustration with Veo overcr...,3/10/2026,Clare Roth,Natalie Weger,bike racks,e-scooters,micromobility,UMD DOTS,veo,Campus
3,Maryland lawmakers consider banning law enforc...,3/9/2026,Nicole Pilsbury,Marijke Friedman,face mask,law enforcement,maryland general assembly,nicole williams,police,State
4,UMD SGA plans to eliminate residential represe...,3/9/2026,Katherine Schutzman,Marijke Friedman,Dhruvak Mirani,residence hall association,sga eleciton,student government association,sga elections commission,Campus
5,Meet the 4 Prince George’s County officials ru...,3/9/2026,Brenna Tichy,Mayah Nachman,5th district,candidates,election,prince george's county,steny hoyer,Local
6,UPDATED: Power returns to all UMD buildings af...,3/8/2026,Pera Onal,Mayah Nachman,dorms,campus,electricity,power outage,power outages,Campus
7,Plans for ICE detention center in Hagerstown s...,3/6/2026,Nicole Pilsbury,Natalie Weger,detention centers,immigration,hagerstown,ice,immigration,State
8,UMD sees boost to program that helps employees...,3/6/2026,Clare Roth,Natalie Weger,employees,public transit,transportation,teaching assistants,pre-tax transit commuter benefits program,Campus
9,UMD SGA demands USM regent resign for his asso...,3/5/2026,Katherine Schutzman,Marijke Friedman,board of regents,epstein files,student government association,tom mcmillen,university system of maryland,Campus


In [30]:
class Author(BaseModel):
    name = CharField(unique=True)

class Tag(BaseModel):
    name = CharField(unique=True)

class Headline(BaseModel):
    title = CharField()
    date = DateField()
    author = ForeignKeyField(Author, backref='headlines')
    category = CharField()

class HeadlineTag(BaseModel):
    headline = ForeignKeyField(Headline, backref='headline_tags')
    tag = ForeignKeyField(Tag, backref='headline_tags')

In [31]:
# connecting to our dbk.db database
dbk.connect()
# drop tables if exist
dbk.drop_tables([Author, Tag, Headline, HeadlineTag], safe=True)
# telling it to create the tables
dbk.create_tables([Author, Tag, Headline, HeadlineTag])

In [32]:
import datetime

# Populate the database from the CSV
for index, row in dbk_stories.iterrows():
    author, _ = Author.get_or_create(name=row['Writer'])
    date_obj = datetime.datetime.strptime(row['Date'], '%m/%d/%Y').date()
    headline = Headline.create(
        title=row['Article'],
        date=date_obj,
        author=author,
        category=row['Category']
    )
    # Create tags
    for tag_col in ['Tag 1', 'Tag 2', 'Tag 3', 'Tage 4 ', 'Tag 5']:
        if pd.notna(row[tag_col]) and row[tag_col].strip():
            tag, _ = Tag.get_or_create(name=row[tag_col].strip())
            HeadlineTag.create(headline=headline, tag=tag)

In [33]:
# Let's check if the data was inserted
print("Authors:")
for author in Author.select():
    print(f"- {author.name}")

print("\nTags:")
for tag in Tag.select():
    print(f"- {tag.name}")

print("\nHeadlines:")
for headline in Headline.select():
    tags = [ht.tag.name for ht in headline.headline_tags]
    print(f"- {headline.title} by {headline.author.name}, tags: {', '.join(tags)}")

Authors:
- Johana Gonzalez-Cruz
- Amelia Twyman
- Clare Roth
- Nicole Pilsbury
- Katherine Schutzman
- Brenna Tichy
- Pera Onal
- Tyler Quattrin
- Caroline McDonough
- Irit Skulnik
- Charlotte Sutton
- Natalie Weger

Tags:
- 251 north
- composting
- umd dining services
- food waste
- recycling
- apartment
- apartment building
- college park housing
- the arbor
- leasing
- bike racks
- e-scooters
- micromobility
- UMD DOTS
- veo
- face mask
- law enforcement
- maryland general assembly
- nicole williams
- police
- Dhruvak Mirani
- residence hall association
- sga eleciton
- student government association
- sga elections commission
- 5th district
- candidates
- election
- prince george's county
- steny hoyer
- dorms
- campus
- electricity
- power outage
- power outages
- detention centers
- immigration
- hagerstown
- ice
- employees
- public transit
- transportation
- teaching assistants
- pre-tax transit commuter benefits program
- board of regents
- epstein files
- tom mcmillen
- unive